# Session 5 — Markov decision processes for an RL-based agent

**Salih Tutun, PhD** · DAT 5668 · Module 3 · Concept session (about 80 minutes)**

I am Salih Tutun. I talk. You guess to a neighbor. Until I say **open the notebook**, this file stays closed. **Do not scroll.** One cell at a time. Two voices, then type. After we run: **LLM**, **RL**, or **hybrid** — and one line for your project.

**Family this session: RL-based agent.** This is the loop of the course, written as an MDP: state, action, environment, reward or penalty, next state. Use an LLM-based agent only for the part that arrives as language. A DAT 5565 model may help form the state. Session 6 writes both designs for one process.

![An RL-based agent, still the loop](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-mdp.png)

![Inventory as a Markov decision process](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/mdp-inventory.png)

## Run of show
- 10 min — Recap bandits. Cold call: what did a bandit not model that inventory needs?
- 25 min — States, actions, transitions, rewards, discounting.
- 30 min — Live inventory MDP. One transition, then a one-step value.
- 15 min — When RL is the wrong recommendation. Session 6's memo uses this test.

## Learning goals
- Write the course loop as an MDP: state, action, environment, reward or penalty, next state.
- Write an MDP for a repeated operations decision.
- Explain discounting as "later cash is worth less, and later mistakes still count."
- List the data you would need before anyone trains a policy.


## How we work in this notebook

I am Salih. I talk. You guess to a neighbor. Two voices share. Then we type **this cell only**. Do not scroll.

| You will see | What you do in class |
| --- | --- |
| **STOP** | Wait. Do not run the next cell until I say. |
| **This cell — ...** | I tell you what we are looking at. Then we run it together. |
| **YOU FILL THIS** | Guess to a neighbor (20 seconds). Two people share. Then type `YOU TYPE HERE`. Then run. |
| **RUN WITH ME** | Already written. Run it. Do not edit. |
| **Family check** | Say **LLM**, **RL**, or **hybrid** out loud — and one line you will keep for your project. |

There is no `TODO` in a concept session. Practice sessions (even numbers) are where you write the function.


## 1. From a bandit to an MDP

A bandit has no memory of stock. A supermarket Home-and-lifestyle shelf does: ordering today changes tomorrow's state. Demand probabilities come from real invoices. That is an MDP. The agent sees s, sends a, the store (the environment) returns r and s', and we repeat.

| Piece | Inventory meaning |
| --- | --- |
| State | Units on hand at the start of the day |
| Action | Units ordered this morning, arriving before demand |
| Transition | Next stock = current + order − demand, not below 0, not above capacity |
| Reward | Margin on units sold − holding cost − stockout penalty |
| Discount γ | How much tomorrow's reward counts today (0.95 in the demo) |

RL is appropriate when the decision **repeats**, the reward can be **defined**, and you can survive the **exploration**. A one-time contract review fails this test.


In [ ]:
# Home-and-lifestyle replenishment. Demand comes from real supermarket invoices.
from pathlib import Path
from collections import Counter, defaultdict
import csv

CAPACITY = 4
HOLD_COST = 0.2
MARGIN = 2.0
STOCKOUT_PENALTY = 1.5


def course_data(name):
    for base in (Path("data"), Path("../../assets/data"), Path("assets/data"), Path("/content/data")):
        path = base / name
        if path.exists():
            return path
    raise FileNotFoundError(name)


def demand_from_invoices():
    """Bin daily Home-and-lifestyle quantity into {0, 1, 2} so the MDP stays readable."""
    daily = defaultdict(int)
    with course_data("supermarket-sales.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if (row.get("Product line") or "").strip() != "Home and lifestyle":
                continue
            daily[row["Date"]] += int(float(row["Quantity"]))
    totals = sorted(daily.values())
    if not totals:
        return {0: 1 / 3, 1: 1 / 3, 2: 1 / 3}
    q1, q2 = totals[len(totals) // 3], totals[(2 * len(totals)) // 3]
    bins = [0 if qty <= q1 else 1 if qty <= q2 else 2 for qty in totals]
    counts = Counter(bins)
    n = len(bins)
    return {level: counts[level] / n for level in (0, 1, 2)}


DEMAND_P = demand_from_invoices()


def next_state(stock, order, demand):
    started = min(CAPACITY, stock + order)
    sales = min(started, demand)
    ending = started - sales
    return ending, sales


def reward(stock, order, demand):
    ending, sales = next_state(stock, order, demand)
    stockout = max(0, demand - min(CAPACITY, stock + order))
    return MARGIN * sales - HOLD_COST * ending - STOCKOUT_PENALTY * stockout


def expected_reward(stock, order):
    """Sum over demand of P(demand) * reward. This is the one-step backup."""
    total = 0.0
    for demand, prob in DEMAND_P.items():
        total += prob * reward(stock, order, demand)
    return total


print("Empirical P(demand) from Home-and-lifestyle invoices:", {k: round(v, 3) for k, v in DEMAND_P.items()})
print("Expected one-step reward from stock=1")
for order in range(3):
    print(f"  order {order}: {expected_reward(1, order):.3f}")


## 2. What "Markov" buys you

Tomorrow's stock depends on today's stock, today's order, and today's demand. It does not depend on last month's path once today's stock is known. If leftover stock spoils based on its age, stock level alone is **not** a Markov state. You would add age to the state.

## 3. Data you would need before training

- A state you can measure before the action (on-hand, not "how the week felt").
- The actions the warehouse can actually perform.
- A reward the CFO agrees with, including the stockout penalty.
- Enough repeated days, or a simulator built from those days.

If any row is missing, Session 6's memo should say **do not automate** or **hybrid** (a person handles the unmeasured part).


In [ ]:
# One simulated week under a base-stock rule: order up to 2, but never above capacity.
def base_stock_order(stock, target=2):
    return max(0, min(2, target - stock))


stock = 1
print("day stock order demand sales end reward")
rng_demands = [1, 2, 0, 2, 1]
for day, demand in enumerate(rng_demands, start=1):
    order = base_stock_order(stock)
    ending, sales = next_state(stock, order, demand)
    pay = reward(stock, order, demand)
    print(f"{day:3} {stock:5} {order:5} {demand:6} {sales:5} {ending:3} {pay:7.2f}")
    stock = ending


## Bridge to Session 6

Students map **one** process twice: as an LLM-agent design (tools, knowledge, handoffs) and as an MDP (states, actions, rewards). The deliverable is a go/no-go memo: LLM, RL, hybrid, or do not automate.

The process they will use, unless you assign another, is **replenishment exceptions**: the daily order is numeric, but a promo email or a supplier delay is a document a person or an LLM agent must read.


## Fill this with me — type this in class

**STOP. Do not scroll.** Neighbor 20s. Two voices. Stock is empty, you order nothing, and two customers show up. Guess the reward, then type, then run.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. A number. Two missed sales times the 1.5 stockout penalty.
expected_reward_guess = None  # YOU TYPE HERE
actual = reward(0, 0, 2)
print("You expected", expected_reward_guess, "| reward() returned", round(actual, 2))
print("The stockout penalty is the business talking. Change STOCKOUT_PENALTY above and rerun this cell.")


## Add this to your framework — STOP. Do not scroll.

Guess to a neighbor (20 seconds): after today, is **your** project **LLM**, **RL**, or **hybrid**? Two voices share. Then type one sentence you will keep.

Hint: if tomorrow's stock depends on today's order, you need an **MDP** (RL family), not a bandit. A promo email is still **LLM** or a person.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. Two voices. Then type.
framework_family = ""  # YOU TYPE HERE: LLM, RL, or hybrid
framework_line = ""    # YOU TYPE HERE: one sentence you will keep
print("Family:", framework_family or "(say it, then type it)")
print("I will keep:", framework_line or "(one sentence)")
